# 문서 벡터화 Document Vectorization
- BOW
- TF-IDF
- DTM / TDM

1. BOW로 단어 빈도 기반 표현을 만든다.
2. 그 결과를 문서-단어 행렬(DTM) 형태로 확인한다.
3. TF-IDF로 단순 빈도가 아니라 단어의 중요도를 반영한다.
4. 벡터 간 코사인 유사도를 구해 문서 간 유사성을 비교한다.

BOW와 TF-IDF 모두 전통적인 벡터화 방식으로,
- 단어의 의미 자체를 깊게 이해하지는 못하고
- 기본적으로 문맥과 어순 반영이 약하며
- 벡터가 희소(sparse)해지기 쉽다는 한계가 있다.

## BOW Bag of Words

`CountVectorizer` 클래스를 통해 텍스트를 토큰화하고, 단어 빈도수 기반으로 특성 벡터를 생성한다.

Bag of Words의 핵심은 문서를 "어떤 단어가 몇 번 나왔는가"로 표현하는 것이다.

장점
- 구조가 단순하고 이해하기 쉽다.
- 빠르게 baseline을 만들기 좋다.
- 전통적인 문서 분류, 검색, 유사도 계산에서 여전히 자주 사용된다.

한계
- 단어 순서를 반영하지 못한다.
- 문맥과 의미를 충분히 반영하지 못한다.
- 단어 수가 많아질수록 벡터 차원이 커지고 대부분이 0인 희소 벡터가 되기 쉽다.


In [1]:
sentences = [
    'I love my dog.',
    'I love my cat.',
    'I love my dog and love my cat.',
    'You love my dog!',
    'Do you think my dog is amazing?'
]

In [ ]:
# %pip install scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ------------------ --------------------- 3.7/8.0 MB 19.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 23.7 MB/s  0:00:00

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   --------------------

In [3]:
from sklearn.feature_extraction.text import CountVectorizer

# 기본적으로 소문자화, 토큰화, 구두점 제거 등을 수행한 뒤 각 문서를 "단어 빈도 벡터"로 반환한다.
vectorizer = CountVectorizer()

# fit : 전체 문서를 보고 단어 사전을 만든다.
# transform : 각 문장을 단어 빈도 벡터로 변한한다.
features = vectorizer.fit_transform(sentences)

In [4]:
# 내용 확인
features.toarray()

array([[0, 0, 0, 0, 1, 0, 1, 1, 0, 0],
       [0, 0, 1, 0, 0, 0, 1, 1, 0, 0],
       [0, 1, 1, 0, 1, 0, 2, 2, 0, 0],
       [0, 0, 0, 0, 1, 0, 1, 1, 0, 1],
       [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]], dtype=int64)

In [5]:
# 단어 사전
features_names = vectorizer.get_feature_names_out()
features_names

array(['amazing', 'and', 'cat', 'do', 'dog', 'is', 'love', 'my', 'think',
       'you'], dtype=object)

In [6]:
import pandas as pd

bow_df = pd.DataFrame(features.toarray(),
                      columns=features_names,
                      index=['sent1','sent2','sent3','sent4','sent5'])
bow_df

,amazing,and,cat,do,dog,is,love,my,think,you
sent1,0,0,0,0,1,0,1,1,0,0
sent2,0,0,1,0,0,0,1,1,0,0
sent3,0,1,1,0,1,0,2,2,0,0
sent4,0,0,0,0,1,0,1,1,0,1
sent5,1,0,0,1,1,1,0,1,1,1


In [7]:
# 각 문서 간의 유사도 구하기
from sklearn.metrics.pairwise import cosine_similarity

bow_sim = cosine_similarity(bow_df)
bow_sim_df = pd.DataFrame(bow_sim,columns=sentences,index=sentences)
bow_sim_df

,I love my dog.,I love my cat.,I love my dog and love my cat.,You love my dog!,Do you think my dog is amazing?
I love my dog.,1.000000,0.666667,0.870388,0.866025,0.436436
I love my cat.,0.666667,1.000000,0.870388,0.577350,0.218218
I love my dog and love my cat.,0.870388,0.870388,1.000000,0.753778,0.341882
You love my dog!,0.866025,0.577350,0.753778,1.000000,0.566947
Do you think my dog is amazing?,0.436436,0.218218,0.341882,0.566947,1.000000


## DTM | TDM

- DTM(Document-Term Matrix): 행(row)은 문서, 열(column)은 단어인 행렬
- TDM(Term-Document Matrix): 행(row)은 단어, 열(column)은 문서인 행렬

중요한 점:
- DTM은 BOW와 완전히 다른 새로운 알고리즘이 아니라,
  BOW 결과를 여러 문서 기준의 표(행렬) 형태로 정리한 표현이라고 이해하면 된다.
- 즉, "단어 빈도 기반 문서 표현"을 보기 쉽게 펼친 형태가 DTM이다.


In [8]:
dtm = pd.DataFrame(
    features.toarray(),
    columns=features_names,
    index=['sent1','sent2','sent3','sent4','sent5']
    )
dtm

,amazing,and,cat,do,dog,is,love,my,think,you
sent1,0,0,0,0,1,0,1,1,0,0
sent2,0,0,1,0,0,0,1,1,0,0
sent3,0,1,1,0,1,0,2,2,0,0
sent4,0,0,0,0,1,0,1,1,0,1
sent5,1,0,0,1,1,1,0,1,1,1


In [ ]:
tdm = dtm.T     # transpose()
tdm

,sent1,sent2,sent3,sent4,sent5
amazing,0,0,0,0,1
and,0,0,1,0,0
cat,0,1,1,0,0
do,0,0,0,0,1
dog,1,0,1,1,1
is,0,0,0,0,1
love,1,1,2,1,0
my,1,1,2,1,1
think,0,0,0,0,1
you,0,0,0,1,1


## TF-IDF

`TfidfVectorizer`는 TF-IDF(Term Frequency - Inverse Document Frequency)라는 가중치를 사용한다.

핵심 아이디어는 다음과 같다.

- 어떤 문서 안에서 자주 등장하는 단어는 그 문서에서 중요할 가능성이 높다.
- 하지만 모든 문서에서 흔하게 등장하는 단어는 구분력이 낮다.
- 따라서 "이 문서 안에서는 자주 나오지만, 전체 문서에서는 흔하지 않은 단어"에 더 높은 가중치를 준다.

즉, CountVectorizer가 단순 빈도를 본다면, TF-IDF는 단어의 중요도를 반영한다.

예를 들어,
- `love`, `my`처럼 여러 문서에 반복적으로 등장하는 단어는 중요도가 상대적으로 낮아질 수 있다.
- 특정 문서에서만 두드러지는 단어는 더 높은 가중치를 가질 수 있다.

**용어**
- $tf(t, d)$: 특정 단어 $t$가 문서 $d$에서 등장한 빈도 (Term Frequency)
- $df(t)$: 특정 단어 $t$가 등장한 문서의 수 (Document Frequency)
- $N$: 전체 문서의 수

### 1. TF (Term Frequency)
단어 $t$가 문서 $d$ 안에서 얼마나 자주 등장하는지를 나타낸다.

$
tf(t, d) = \frac{\text{단어 } t \text{의 문서 } d \text{ 내 등장 횟수}}{\text{문서 } d \text{의 전체 단어 수}}
$

### 2. IDF (Inverse Document Frequency)
단어가 전체 문서 집합에서 얼마나 희귀한지를 나타낸다.

- 여러 문서에서 공통으로 많이 등장하는 단어라면 IDF 값은 작아진다.
- 적은 문서에서만 등장하는 단어라면 IDF 값은 커진다.

$
idf(t) = \log\left(\frac{1 + N}{1 + df(t)}\right) + 1
$

여기서 $1$을 더하는 이유는 분모가 0이 되는 상황을 피하고, 계산을 안정적으로 만들기 위해서이다.

### 3. 최종 TF-IDF
최종 가중치는 다음과 같이 계산된다.

$
tfidf(t, d) = tf(t, d) \times idf(t)
$

정리하면,
- TF는 "이 문서 안에서 얼마나 자주 나오는가"
- IDF는 "전체 문서에서 얼마나 드문가"
- TF-IDF는 "이 문서를 특징짓는 단어인가"

를 수치화한 것이다.


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

# Count 기반 문서 표현에 IDF 가중치를 반영한 결과
features = tfidf_vectorizer.fit_transform(sentences)
print(features)     # 회소 배열

features = features.toarray()       # 밀집 배열로 변환
features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 22 stored elements and shape (5, 10)>
  Coords	Values
  (0, 6)	0.6068561362933035
  (0, 7)	0.5132750331803866
  (0, 4)	0.6068561362933035
  (1, 6)	0.5152898800248592
  (1, 7)	0.43582888010783327
  (1, 2)	0.7379224395611763
  (2, 6)	0.5533643125368353
  (2, 7)	0.4680319912607929
  (2, 4)	0.27668215626841763
  (2, 2)	0.3962235232075343
  (2, 1)	0.4911088441748528
  (3, 6)	0.4580537876334307
  (3, 7)	0.3874189597587254
  (3, 4)	0.4580537876334307
  (3, 9)	0.6559573194109529
  (4, 7)	0.2090544457153013
  (4, 4)	0.2471695777128123
  (4, 9)	0.35395994534638453
  (4, 3)	0.4387242287788316
  (4, 8)	0.4387242287788316
  (4, 5)	0.4387242287788316
  (4, 0)	0.4387242287788316


array([[0.        , 0.        , 0.        , 0.        , 0.60685614,
        0.        , 0.60685614, 0.51327503, 0.        , 0.        ],
       [0.        , 0.        , 0.73792244, 0.        , 0.        ,
        0.        , 0.51528988, 0.43582888, 0.        , 0.        ],
       [0.        , 0.49110884, 0.39622352, 0.        , 0.27668216,
        0.        , 0.55336431, 0.46803199, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.45805379,
        0.        , 0.45805379, 0.38741896, 0.        , 0.65595732],
       [0.43872423, 0.        , 0.        , 0.43872423, 0.24716958,
        0.43872423, 0.        , 0.20905445, 0.43872423, 0.35395995]])

In [13]:
# 컬럼(열) : 토큰
features_names = tfidf_vectorizer.get_feature_names_out()
features_names

array(['amazing', 'and', 'cat', 'do', 'dog', 'is', 'love', 'my', 'think',
       'you'], dtype=object)

In [14]:
sent_DF = pd.DataFrame(features,
                       columns=features_names,
                       index=['sent1','sent2','sent3','sent4','sent5'])
sent_DF

,amazing,and,cat,do,dog,is,love,my,think,you
sent1,0.000000,0.000000,0.000000,0.000000,0.606856,0.000000,0.606856,0.513275,0.000000,0.000000
sent2,0.000000,0.000000,0.737922,0.000000,0.000000,0.000000,0.515290,0.435829,0.000000,0.000000
sent3,0.000000,0.491109,0.396224,0.000000,0.276682,0.000000,0.553364,0.468032,0.000000,0.000000
sent4,0.000000,0.000000,0.000000,0.000000,0.458054,0.000000,0.458054,0.387419,0.000000,0.655957
sent5,0.438724,0.000000,0.000000,0.438724,0.247170,0.438724,0.000000,0.209054,0.438724,0.353960


In [15]:
# 각 문서 간의 유사도 구하기
from sklearn.metrics.pairwise import cosine_similarity

sent_sim = cosine_similarity(sent_DF)
sent_sim_df = pd.DataFrame(sent_sim,columns=sentences,index= sentences)
sent_sim_df

,I love my dog.,I love my cat.,I love my dog and love my cat.,You love my dog!,Do you think my dog is amazing?
I love my dog.,1.000000,0.536407,0.743948,0.754798,0.257299
I love my cat.,0.536407,1.000000,0.781507,0.404879,0.091112
I love my dog and love my cat.,0.743948,0.781507,1.000000,0.561530,0.166232
You love my dog!,0.754798,0.404879,0.561530,1.000000,0.426391
Do you think my dog is amazing?,0.257299,0.091112,0.166232,0.426391,1.000000
